In [ ]:
# EDA and Modeling with PCA & K-Fold Validation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.decomposition import PCA
sns.set_theme(style="whitegrid")

In [ ]:
# 1. Load Data
df = pd.read_csv('../data/divorce.csv', sep=';')
if len(df.columns) == 1:
    df = pd.read_csv('../data/divorce.csv', sep=',')
df.dropna(inplace=True)
if 'Id' in df.columns:
    df.drop('Id', axis=1, inplace=True)

In [ ]:
# 2. Train/Test Split (FIRST to avoid Data Leakage)
X_all = df.drop('Class', axis=1)
y = df['Class']
X_train_all, X_test_all, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42)

In [ ]:
# 3. Exploratory Data Analysis
plt.figure(figsize=(20, 15))
df.hist(bins=15, figsize=(20, 15), layout=(8, 7))
plt.tight_layout()
plt.show()

In [ ]:
# PCA to visually prove linearly separable dataset
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_all)
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, palette='Set1', s=100)
plt.title('PCA of Divorce Dataset (Showing Perfect Separability)')
plt.show()

In [ ]:
# 4. Feature Selection (on TRAIN SET ONLY to prevent leakage)
train_df = pd.concat([X_train_all, y_train], axis=1)
corr_matrix_train = train_df.corr()
plt.figure(figsize=(20, 15))
sns.heatmap(corr_matrix_train, cmap='coolwarm', annot=False, fmt=".2f")
plt.title("Correlation Heatmap (Train Set Only)")
plt.show()

In [ ]:
# Select top 10 features from train set only
top_features_list = corr_matrix_train['Class'].sort_values(ascending=False).head(11).index.tolist()
top_features_list.remove('Class')
X_train_top = X_train_all[top_features_list]
X_test_top = X_test_all[top_features_list]

In [ ]:
# 5. Model Training (LogRes All, LogRes Top, RF Top)
logres_all = LogisticRegression(max_iter=1000, random_state=42)
logres_all.fit(X_train_all, y_train)

logres_fs = LogisticRegression(max_iter=1000, random_state=42)
logres_fs.fit(X_train_top, y_train)

rf_fs = RandomForestClassifier(n_estimators=100, random_state=42)
rf_fs.fit(X_train_top, y_train)

In [ ]:
# 6. Evaluation on Unseen Test Set
def evaluate_model(model, X_test_data, name):
    y_pred = model.predict(X_test_data)
    print(f"--- {name} ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1-Score:", f1_score(y_test, y_pred))
    print()

evaluate_model(logres_all, X_test_all, "Logistic Regression (All Features)")
evaluate_model(logres_fs, X_test_top, "Logistic Regression (Top Features)")
evaluate_model(rf_fs, X_test_top, "Random Forest (Top Features)")

In [ ]:
# 7. Export Models
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(logres_all, '../models/logres_all.pkl')
joblib.dump(logres_fs, '../models/logres_fs.pkl')
joblib.dump(rf_fs, '../models/rf_fs.pkl')
joblib.dump(top_features_list, '../models/top_features.pkl')